# Perfil de qualidade da camada Silver

Verifica completude, consistência, unicidade, acurácia e outliers das nove tabelas da camada Silver.

In [0]:
%sql

-- Contexto e criação da tabela de resultados

USE CATALOG mvp_pipeline;

CREATE OR REPLACE TABLE silver.relatorio_qualidade (
  dimensao     STRING COMMENT 'Completude, Consistência, Unicidade, Acurácia ou Outliers',
  tabela       STRING COMMENT 'Tabela avaliada na camada Silver',
  verificacao  STRING COMMENT 'O que foi verificado',
  valor        DOUBLE COMMENT 'Resultado numérico da verificação',
  esperado     STRING COMMENT 'Critério de aceitação',
  situacao     STRING COMMENT 'OK, ATENÇÃO ou TRATADO',
  observacao   STRING COMMENT 'Interpretação e tratamento adotado',
  _execucao_ts TIMESTAMP
) COMMENT 'Resultado das verificações de qualidade da camada Silver';

## 1. Completude

Valores nulos ou vazios nas colunas que sustentam as análises. 

In [0]:
%sql

-- Mede valores nulos ou vazios nas colunas

INSERT INTO silver.relatorio_qualidade
SELECT 'Completude', 'icms_cnae_subclasse', 'valor_icms nulo',
       sum(CASE WHEN valor_icms IS NULL THEN 1 ELSE 0 END),
       '0', CASE WHEN sum(CASE WHEN valor_icms IS NULL THEN 1 ELSE 0 END) = 0
                 THEN 'OK' ELSE 'ATENÇÃO' END,
       'Falha de conversão do decimal com vírgula apareceria aqui', current_timestamp()
FROM silver.icms_cnae_subclasse
UNION ALL
SELECT 'Completude', 'icms_cnae_subclasse', 'cnae_subclasse nulo ou vazio',
       sum(CASE WHEN cnae_subclasse IS NULL OR cnae_subclasse = '' THEN 1 ELSE 0 END),
       '0', 'OK', 'Linhas sem CNAE aparecem como 0000000, marcadas por sem_cnae', current_timestamp()
FROM silver.icms_cnae_subclasse
UNION ALL
SELECT 'Completude', 'icms_cnae_subclasse', 'percentual do ICMS sem CNAE atribuído',
       round(100 * sum(CASE WHEN sem_cnae THEN valor_icms ELSE 0 END) / sum(valor_icms), 2),
       'informativo', 'TRATADO',
       'A fonte não atribui atividade econômica a essa parcela. Não é corrigível e define o teto de cobertura do recorte setorial: nenhuma cadeia produtiva alcança esse valor',
       current_timestamp()
FROM silver.icms_cnae_subclasse
UNION ALL
SELECT 'Completude', 'arrecadacao_municipio', 'valor_arrecadado nulo',
       sum(CASE WHEN valor_arrecadado IS NULL THEN 1 ELSE 0 END),
       '0', CASE WHEN sum(CASE WHEN valor_arrecadado IS NULL THEN 1 ELSE 0 END) = 0
                 THEN 'OK' ELSE 'ATENÇÃO' END,
       '', current_timestamp()
FROM silver.arrecadacao_municipio
UNION ALL
SELECT 'Completude', 'desoneracoes', 'valor_desonerado nulo',
       sum(CASE WHEN valor_desonerado IS NULL THEN 1 ELSE 0 END),
       '0', CASE WHEN sum(CASE WHEN valor_desonerado IS NULL THEN 1 ELSE 0 END) <= 1
                 THEN 'TRATADO' ELSE 'ATENÇÃO' END,
       'Ausente na origem em 1 de 121.162 linhas (0,0008%): ISENÇÃO 177, medicamentos para fibrose cística, Vale do Rio dos Sinos, 2023. Não é supressão por sigilo: 80.178 das 80.179 linhas com um único contribuinte têm valor publicado. Erro semântico não corrigível, mantido nulo e sem imputação.',
       current_timestamp()
FROM silver.desoneracoes
UNION ALL
SELECT 'Completude', 'desoneracoes', 'cnae_subclasse ausente no nível detalhado',
       sum(CASE WHEN nivel_agregacao = 'DETALHADO' AND (cnae_subclasse IS NULL OR cnae_subclasse = '0000000')
                THEN 1 ELSE 0 END),
       '0', 'OK', 'O nível agregado estadual não tem CNAE por natureza', current_timestamp()
FROM silver.desoneracoes
UNION ALL
SELECT 'Completude', 'cadastro_setor', 'qtd_ativos nulo',
       sum(CASE WHEN qtd_ativos IS NULL THEN 1 ELSE 0 END),
       '0', CASE WHEN sum(CASE WHEN qtd_ativos IS NULL THEN 1 ELSE 0 END) = 0
                 THEN 'OK' ELSE 'ATENÇÃO' END,
       '', current_timestamp()
FROM silver.cadastro_setor
UNION ALL
SELECT 'Completude', 'cadastro_municipio', 'cod_municipio_ibge nulo',
       sum(CASE WHEN cod_municipio_ibge IS NULL THEN 1 ELSE 0 END),
       '0', CASE WHEN sum(CASE WHEN cod_municipio_ibge IS NULL THEN 1 ELSE 0 END) = 0
                 THEN 'OK' ELSE 'ATENÇÃO' END,
       '', current_timestamp()
FROM silver.cadastro_municipio
UNION ALL
SELECT 'Completude', 'pib_municipal', 'pib_reais nulo',
       sum(CASE WHEN pib_reais IS NULL THEN 1 ELSE 0 END),
       '0', CASE WHEN sum(CASE WHEN pib_reais IS NULL THEN 1 ELSE 0 END) = 0
                 THEN 'OK' ELSE 'ATENÇÃO' END,
       '', current_timestamp()
FROM silver.pib_municipal;

In [0]:
%sql

-- Verificação

SELECT * FROM silver.relatorio_qualidade WHERE dimensao = 'Completude' ORDER BY tabela;

## 2. Consistência

Formato e domínio esperados: tamanho de código, faixa de mês, valores negativos e grafia dos domínios textuais.

In [0]:
%sql

-- Consistência de formato e domínio

INSERT INTO silver.relatorio_qualidade
SELECT 'Consistência', 'icms_cnae_subclasse', 'cnae_subclasse fora de 7 dígitos',
       sum(CASE WHEN length(cnae_subclasse) <> 7 THEN 1 ELSE 0 END),
       '0', CASE WHEN sum(CASE WHEN length(cnae_subclasse) <> 7 THEN 1 ELSE 0 END) = 0
                 THEN 'OK' ELSE 'ATENÇÃO' END,
       'Padronização aplicada com lpad na Silver', current_timestamp()
FROM silver.icms_cnae_subclasse
UNION ALL
SELECT 'Consistência', 'cnae_cadeia', 'cnae_subclasse fora de 7 dígitos',
       sum(CASE WHEN length(cnae_subclasse) <> 7 THEN 1 ELSE 0 END),
       '0', 'OK', 'Corrige os 168 códigos que perderam o zero à esquerda na planilha',
       current_timestamp()
FROM silver.cnae_cadeia
UNION ALL
SELECT 'Consistência', 'cadastro_municipio', 'cod_municipio_ibge fora de 7 dígitos',
       sum(CASE WHEN length(cod_municipio_ibge) <> 7 THEN 1 ELSE 0 END),
       '0', 'OK', '', current_timestamp()
FROM silver.cadastro_municipio
UNION ALL
SELECT 'Consistência', 'icms_cnae_subclasse', 'mes fora da faixa 1 a 12',
       sum(CASE WHEN mes < 1 OR mes > 12 THEN 1 ELSE 0 END),
       '0', 'OK', '', current_timestamp()
FROM silver.icms_cnae_subclasse
UNION ALL
SELECT 'Consistência', 'icms_cnae_subclasse', 'valor_icms negativo',
       sum(CASE WHEN valor_icms < 0 THEN 1 ELSE 0 END),
       'informativo', 'TRATADO',
       'Arrecadação líquida negativa por restituição e compensação: 16 linhas em 114.243, somando -R$ 10,4 milhões, ou 0,00002% do total. Mantidas, pois excluí-las superestimaria a arrecadação',
       current_timestamp()
FROM silver.icms_cnae_subclasse
UNION ALL
SELECT 'Consistência', 'desoneracoes', 'valor_desonerado negativo',
       sum(CASE WHEN valor_desonerado < 0 THEN 1 ELSE 0 END),
       '0', CASE WHEN sum(CASE WHEN valor_desonerado < 0 THEN 1 ELSE 0 END) = 0
                 THEN 'OK' ELSE 'ATENÇÃO' END,
       'Desoneração é valor não arrecadado: negativo não teria interpretação', current_timestamp()
FROM silver.desoneracoes
UNION ALL
SELECT 'Consistência', 'cadastro_setor', 'categorias distintas',
       count(DISTINCT categoria), '5 categorias conhecidas', 'OK',
       'MEI, SIMPLES NACIONAL, GERAL, MICROPRODUTOR, PRODUTOR', current_timestamp()
FROM silver.cadastro_setor
UNION ALL
SELECT 'Consistência', 'desoneracoes', 'tipos de benefício distintos',
       count(DISTINCT tipo_beneficio), '10', 'OK',
       'Domínio fechado e estável nos quatro anos', current_timestamp()
FROM silver.desoneracoes
UNION ALL
SELECT 'Consistência', 'desoneracoes', 'tipos de benefício que colidem ao remover acentos',
       CAST(count(DISTINCT tipo_beneficio)
            - count(DISTINCT translate(tipo_beneficio, 'ÃÁÀÂÉÊÍÓÔÕÚÜÇ', 'AAAAEEIOOOUUC')) AS DOUBLE),
       '0', CASE WHEN count(DISTINCT tipo_beneficio)
                    - count(DISTINCT translate(tipo_beneficio, 'ÃÁÀÂÉÊÍÓÔÕÚÜÇ', 'AAAAEEIOOOUUC')) = 0
                 THEN 'OK' ELSE 'ATENÇÃO' END,
       'A fonte grafa NAO INCIDÊNCIA sem til e NÃO ESTORNO DO CRÉDITO com til, mas cada conceito tem grafia única: não há agrupamento partido',
       current_timestamp()
FROM silver.desoneracoes;

In [0]:
%sql

-- Verificação

SELECT tabela, verificacao, valor, esperado, situacao
FROM silver.relatorio_qualidade
WHERE dimensao = 'Consistência'
ORDER BY tabela, verificacao;


## 3. Unicidade

Cada tabela respeita a granularidade declarada. Duplicata silenciosa dobra valores em qualquer
agregação.

Os grãos abaixo foram confirmados por investigação: o ICMS inclui a versão da CNAE, porque a fonte
publica as classificações 1.1 e 2.0 lado a lado.

In [0]:
%sql

-- Unicidade por granularidade declarada

INSERT INTO silver.relatorio_qualidade
SELECT 'Unicidade', 'icms_cnae_subclasse', 'duplicatas no grão declarado',
       count(*), '0', CASE WHEN count(*) = 0 THEN 'OK' ELSE 'ATENÇÃO' END,
       'Grão: ano + mes + versao_cnae + cnae_subclasse + nome_cnae_subclasse. A fonte publica CNAE 1.1 e 2.0 lado a lado e, desde nov/2024, duas categorias residuais sob o mesmo código 0000000 — SEM CNAE e OUTROS, ambas na versão 2.0. São agregados distintos, não repetição',
       current_timestamp()
FROM (SELECT ano, mes, versao_cnae, cnae_subclasse, nome_cnae_subclasse
      FROM silver.icms_cnae_subclasse
      GROUP BY ano, mes, versao_cnae, cnae_subclasse, nome_cnae_subclasse HAVING count(*) > 1)
UNION ALL
SELECT 'Unicidade', 'arrecadacao_municipio', 'duplicatas em ano+mes+municipio+tributo',
       count(*), '0', CASE WHEN count(*) <= 1 THEN 'TRATADO' ELSE 'ATENÇÃO' END,
       'Caso único: Porto Alegre, IPVA, jul/2025, com R$ 890,10 numa linha SEM COREDE ao lado de R$ 37,08 milhões no COREDE correto. Resolvido na Gold, onde o COREDE vem da dimensão de município',
       current_timestamp()
FROM (SELECT ano, mes, cod_munic_sefaz, tributo FROM silver.arrecadacao_municipio
      GROUP BY ano, mes, cod_munic_sefaz, tributo HAVING count(*) > 1)
UNION ALL
SELECT 'Unicidade', 'cadastro_municipio', 'duplicatas em ano+mes+categoria+municipio',
       count(*), '0', CASE WHEN count(*) = 0 THEN 'OK' ELSE 'ATENÇÃO' END,
       '', current_timestamp()
FROM (SELECT ano, mes, cod_categoria, cod_municipio_ibge FROM silver.cadastro_municipio
      GROUP BY ano, mes, cod_categoria, cod_municipio_ibge HAVING count(*) > 1)
UNION ALL
SELECT 'Unicidade', 'pib_municipal', 'duplicatas em municipio+ano',
       count(*), '0', CASE WHEN count(*) = 0 THEN 'OK' ELSE 'ATENÇÃO' END,
       '', current_timestamp()
FROM (SELECT cod_municipio_ibge, ano FROM silver.pib_municipal
      GROUP BY cod_municipio_ibge, ano HAVING count(*) > 1)
UNION ALL
SELECT 'Unicidade', 'municipio_ibge', 'duplicatas de código IBGE',
       count(*), '0', CASE WHEN count(*) = 0 THEN 'OK' ELSE 'ATENÇÃO' END,
       'Dimensão de município deve ter chave única', current_timestamp()
FROM (SELECT cod_municipio_ibge FROM silver.municipio_ibge
      GROUP BY cod_municipio_ibge HAVING count(*) > 1)
UNION ALL
SELECT 'Unicidade', 'cnae_cadeia', 'pares CNAE+cadeia repetidos',
       count(*), '0', CASE WHEN count(*) = 0 THEN 'OK' ELSE 'ATENÇÃO' END,
       'A ponte é N:N; o par não pode repetir', current_timestamp()
FROM (SELECT cnae_subclasse, cadeia FROM silver.cnae_cadeia
      GROUP BY cnae_subclasse, cadeia HAVING count(*) > 1)
UNION ALL
SELECT 'Unicidade', 'cadastro_setor', 'duplicatas em ano+mes+categoria+cnae',
       count(*), '0', CASE WHEN count(*) = 0 THEN 'OK' ELSE 'ATENÇÃO' END,
       'Granularidade confirmada por inspeção', current_timestamp()
FROM (SELECT ano, mes, categoria, cnae_subclasse FROM silver.cadastro_setor
      GROUP BY ano, mes, categoria, cnae_subclasse HAVING count(*) > 1)
UNION ALL
SELECT 'Unicidade', 'desoneracoes', 'grupos com mais de uma linha no grão declarado',
       count(*), 'informativo', 'TRATADO',
       'A fonte publica 16 pares idênticos em todas as colunas exceto o valor (ex.: ISENÇÃO 4, Serra, CNAE 4631100, 2021: R$ 12,59 e R$ 25,08). O arquivo tem grão mais fino que as colunas publicadas revelam. As agregações somam, o que a aderência de 0,37% ao total estadual publicado confirma',
       current_timestamp()
FROM (SELECT ano, imposto, tipo_beneficio, cod_beneficio, nome_corede, cnae_subclasse
      FROM silver.desoneracoes WHERE nivel_agregacao = 'DETALHADO'
      GROUP BY ano, imposto, tipo_beneficio, cod_beneficio, nome_corede, cnae_subclasse
      HAVING count(*) > 1)
UNION ALL
SELECT 'Unicidade', 'desoneracoes', 'dispositivos ambíguos em imposto+tipo+codigo',
       count(*), '0', CASE WHEN count(*) = 0 THEN 'OK' ELSE 'ATENÇÃO' END,
       'Chave da dimensão de benefício: cod_beneficio é sequencial dentro de cada imposto e tipo, nunca global',
       current_timestamp()
FROM (SELECT imposto, tipo_beneficio, cod_beneficio FROM silver.desoneracoes
      GROUP BY imposto, tipo_beneficio, cod_beneficio
      HAVING count(DISTINCT descr_beneficio) > 1);

In [0]:
%sql
-- Verificação

SELECT tabela, verificacao, valor, esperado, situacao
FROM silver.relatorio_qualidade
WHERE dimensao = 'Unicidade'
ORDER BY tabela, verificacao;

## 4. Acurácia

Os valores fazem sentido no contexto. Duas verificações são aferições contra números publicados pela
própria Receita Estadual, o que as torna independentes: o crédito presumido de 2024 e o total
estadual de desonerações de 2023.

In [0]:
%sql

-- Ordens de grandeza do ICMS e do PIB

-- ICMS total por ano: esperado dezenas de bilhões
SELECT ano, sum(valor_icms) AS icms_cnae, count(DISTINCT mes) AS meses
FROM silver.icms_cnae_subclasse GROUP BY ano ORDER BY ano;

-- comparação entre as duas séries de arrecadação do mesmo tributo
SELECT c.ano, c.icms_por_cnae, m.icms_por_municipio,
       round(100 * (c.icms_por_cnae - m.icms_por_municipio) / m.icms_por_municipio, 2) AS dif_perc
FROM (SELECT ano, sum(valor_icms) AS icms_por_cnae
      FROM silver.icms_cnae_subclasse GROUP BY ano) c
JOIN (SELECT ano, sum(valor_arrecadado) AS icms_por_municipio
      FROM silver.arrecadacao_municipio WHERE tributo = 'ICMS' GROUP BY ano) m
  ON c.ano = m.ano
ORDER BY c.ano;

-- PIB de Porto Alegre: esperado entre R$ 80 e 110 bilhões nos anos recentes
-- filtra pelo código IBGE, não pelo nome: a API devolve o nome com sufixo de UF
SELECT ano, nome_municipio, pib_reais FROM silver.pib_municipal
WHERE cod_municipio_ibge = '4314902' ORDER BY ano;

-- PIB estadual: soma dos municípios, esperado em torno de R$ 600 bilhões em 2023
SELECT ano, sum(pib_reais) AS pib_rs, count(*) AS municipios
FROM silver.pib_municipal GROUP BY ano ORDER BY ano;

In [0]:
%sql

-- Acurácia do PIB e consistência de nomes entre as tabelas do IBGE

INSERT INTO silver.relatorio_qualidade
SELECT 'Acurácia', 'pib_municipal', 'PIB de Porto Alegre em 2023 (R$ bilhões)',
       round(pib_reais/1e9, 1), 'entre 80 e 110', 'OK',
       'Capital responde por 16,1% do PIB estadual', current_timestamp()
FROM silver.pib_municipal WHERE ano = 2023 AND cod_municipio_ibge = '4314902';

INSERT INTO silver.relatorio_qualidade
SELECT 'Consistência', 'pib_municipal x municipio_ibge', 'nomes de município divergentes entre as duas tabelas do IBGE',
       CAST(count(*) AS DOUBLE), '0', CASE WHEN count(*) = 0 THEN 'OK' ELSE 'ATENÇÃO' END,
       'A API de agregados devolve o nome com sufixo de UF e a de localidades sem; normalizado na Silver',
       current_timestamp()
FROM silver.pib_municipal p
JOIN silver.municipio_ibge m ON m.cod_municipio_ibge = p.cod_municipio_ibge
WHERE p.nome_municipio <> m.nome_municipio;

INSERT INTO silver.relatorio_qualidade
SELECT 'Acurácia', 'pib_municipal', 'PIB do RS em 2023 (R$ bilhões)',
       round(sum(pib_reais)/1e9, 1), 'aprox. 650', 'OK',
       'Ordem de grandeza aderente ao IBGE. Valores a preços correntes: a série nominal não mede crescimento real',
       current_timestamp()
FROM silver.pib_municipal WHERE ano = 2023;

In [0]:
%sql

-- Comparação com os valores publicados nas notas técnicas da Receita Estadual

SELECT imposto, nivel_agregacao,
       sum(valor_desonerado) / 1e9 AS valor_bi
FROM silver.desoneracoes
WHERE ano = 2023
GROUP BY imposto, nivel_agregacao ORDER BY imposto, nivel_agregacao;

SELECT ano, round(sum(valor_desonerado)/1e6, 2) AS valor_milhoes
FROM silver.desoneracoes
WHERE upper(tipo_beneficio) LIKE '%PRESUMIDO%' AND ano BETWEEN 2021 AND 2024
GROUP BY ano ORDER BY ano;

SELECT
  round(sum(valor_desonerado)/1e9, 3) AS total_na_base_bi,
  round(sum(CASE WHEN upper(tipo_beneficio) IN ('IMUNIDADE','NAO INCIDÊNCIA','SIMPLES NACIONAL')
                   OR upper(finalidade) = 'NEC EXPORTAÇÕES'
              THEN valor_desonerado ELSE 0 END)/1e9, 3) AS heteronomas_bi,
  round(sum(CASE WHEN upper(tipo_beneficio) IN ('IMUNIDADE','NAO INCIDÊNCIA','SIMPLES NACIONAL')
                   OR upper(finalidade) = 'NEC EXPORTAÇÕES'
              THEN 0 ELSE valor_desonerado END)/1e9, 3) AS total_estadual_bi
FROM silver.desoneracoes WHERE ano = 2023;

In [0]:
%sql

-- Registro das aferições e da dupla contagem de exportações

INSERT INTO silver.relatorio_qualidade
SELECT 'Acurácia', 'desoneracoes', 'total estadual de 2023 x valor publicado (R$ 15 bi)',
       round(sum(CASE WHEN upper(tipo_beneficio) IN ('IMUNIDADE','NAO INCIDÊNCIA','SIMPLES NACIONAL')
                        OR upper(finalidade) = 'NEC EXPORTAÇÕES'
                   THEN 0 ELSE valor_desonerado END)/1e9, 3),
       'aprox. 15', 'OK',
       'Confere a 0,37% após excluir as desonerações heterônomas, que a fonte não soma ao total do Estado',
       current_timestamp()
FROM silver.desoneracoes WHERE ano = 2023;

INSERT INTO silver.relatorio_qualidade
SELECT 'Consistência', 'desoneracoes', 'anos em que EXPORTAÇÕES difere de NEC EXPORTAÇÕES',
       CAST(count(*) AS DOUBLE), '0', 'TRATADO',
       'Valores idênticos nos quatro anos; as duas linhas descrevem o mesmo bloco por óticas diferentes e nunca são somadas',
       current_timestamp()
FROM (
  SELECT ano
  FROM silver.desoneracoes
  WHERE upper(finalidade) IN ('EXPORTAÇÕES','NEC EXPORTAÇÕES')
  GROUP BY ano
  HAVING round(sum(CASE WHEN upper(finalidade) = 'EXPORTAÇÕES' THEN valor_desonerado ELSE 0 END), 0)
      <> round(sum(CASE WHEN upper(finalidade) = 'NEC EXPORTAÇÕES' THEN valor_desonerado ELSE 0 END), 0)
);

INSERT INTO silver.relatorio_qualidade
SELECT 'Acurácia', 'icms_cnae_subclasse x arrecadacao_municipio',
       'diferença percentual entre as duas séries de ICMS em 2024',
       round(100 * (c.v - m.v) / m.v, 2), 'até 5%',
       CASE WHEN abs(100 * (c.v - m.v) / m.v) <= 5 THEN 'OK' ELSE 'ATENÇÃO' END,
       'Séries independentes do mesmo tributo, recortes diferentes', current_timestamp()
FROM (SELECT sum(valor_icms) AS v FROM silver.icms_cnae_subclasse WHERE ano = 2024) c,
     (SELECT sum(valor_arrecadado) AS v FROM silver.arrecadacao_municipio
      WHERE tributo = 'ICMS' AND ano = 2024) m;

## 5. Outliers

Em dados fiscais, outlier costuma ser informação legítima.
O objetivo é conhecê-los antes de interpretar qualquer média.

Inclui o marcador dos cinco dispositivos criados para a calamidade de abril e maio de 2024, que
tornam aquele ano atípico nas cadeias de veículos, transporte, comércio e alimentos.

In [0]:
%sql

-- Concentração, distribuição e maiores registros

-- concentração: quanto as 10 maiores subclasses representam do ICMS de 2024
WITH por_cnae AS (
  SELECT cnae_subclasse, nome_cnae_subclasse, sum(valor_icms) AS valor
  FROM silver.icms_cnae_subclasse WHERE ano = 2024 GROUP BY 1, 2
),
total AS (SELECT sum(valor) AS t FROM por_cnae)
SELECT p.cnae_subclasse, p.nome_cnae_subclasse, p.valor,
       round(100 * p.valor / t.t, 2) AS perc_do_total
FROM por_cnae p, total t ORDER BY p.valor DESC LIMIT 10;

-- distribuição: percentis do valor mensal por subclasse em 2024
SELECT percentile(valor_icms, 0.50) AS mediana,
       percentile(valor_icms, 0.90) AS p90,
       percentile(valor_icms, 0.99) AS p99,
       max(valor_icms) AS maximo
FROM silver.icms_cnae_subclasse WHERE ano = 2024 AND valor_icms > 0;

-- desonerações: maiores registros do nível detalhado
SELECT ano, nome_corede, cnae_subclasse, nome_cnae_subclasse, tipo_beneficio, valor_desonerado
FROM silver.desoneracoes WHERE nivel_agregacao = 'DETALHADO'
ORDER BY valor_desonerado DESC LIMIT 10;

In [0]:
%sql

-- Dispositivos extraordinários da calamidade de 2024

SELECT tipo_beneficio, cod_beneficio, descr_beneficio, ano,
       round(sum(valor_desonerado)/1e6, 2) AS valor_milhoes,
       count(DISTINCT cnae_subclasse) AS subclasses,
       count(DISTINCT nome_corede)   AS coredes
FROM silver.desoneracoes
WHERE imposto = 'ICMS'
  AND ((tipo_beneficio = 'ISENÇÃO'           AND cod_beneficio IN (188, 189, 192, 194))
    OR (tipo_beneficio = 'CRÉDITO PRESUMIDO' AND cod_beneficio = 235))
GROUP BY 1, 2, 3, 4 ORDER BY valor_milhoes DESC;

In [0]:
%sql

-- Registro da concentração e da assimetria

INSERT INTO silver.relatorio_qualidade
SELECT 'Outliers', 'icms_cnae_subclasse', 'razão entre máximo e mediana do valor mensal por subclasse em 2024',
       round(max(valor_icms) / percentile(valor_icms, 0.50), 0), 'informativo', 'TRATADO',
       'Distribuição extremamente assimétrica: análises usam participação no total e mediana, nunca média por subclasse',
       current_timestamp()
FROM silver.icms_cnae_subclasse WHERE ano = 2024 AND valor_icms > 0;

INSERT INTO silver.relatorio_qualidade
SELECT 'Outliers', 'icms_cnae_subclasse', 'participação das 10 maiores subclasses no ICMS de 2024',
       round(100 * (SELECT sum(valor) FROM (
         SELECT sum(valor_icms) AS valor FROM silver.icms_cnae_subclasse
         WHERE ano = 2024 GROUP BY cnae_subclasse ORDER BY valor DESC LIMIT 10))
         / (SELECT sum(valor_icms) FROM silver.icms_cnae_subclasse WHERE ano = 2024), 2),
       'informativo', 'OK',
       'Concentração alta é característica do ICMS, não erro; afeta a leitura de médias',
       current_timestamp();

## 6. Integridade temporal

In [0]:
%sql

-- Meses por ano em cada série mensal

SELECT 'icms_cnae_subclasse' AS tabela, ano, count(DISTINCT mes) AS meses
FROM silver.icms_cnae_subclasse GROUP BY ano
UNION ALL
SELECT 'arrecadacao_municipio', ano, count(DISTINCT mes)
FROM silver.arrecadacao_municipio GROUP BY ano
UNION ALL
SELECT 'cadastro_setor', ano, count(DISTINCT mes)
FROM silver.cadastro_setor GROUP BY ano
ORDER BY tabela, ano;

In [0]:
%sql

-- Registro dos anos incompletos e da janela comum às fontes

INSERT INTO silver.relatorio_qualidade
SELECT 'Consistência', 'todas as séries mensais', 'anos com menos de 12 meses',
       CAST(count(*) AS DOUBLE), 'apenas o ano corrente',
       CASE WHEN count(*) <= 1 THEN 'TRATADO' ELSE 'ATENÇÃO' END,
       'Só 2026 está incompleto; anos parciais são excluídos das comparações anuais na Gold',
       current_timestamp()
FROM (SELECT ano FROM silver.icms_cnae_subclasse GROUP BY ano HAVING count(DISTINCT mes) < 12);

INSERT INTO silver.relatorio_qualidade
SELECT 'Consistência', 'todas as fontes', 'anos completos comuns a todas as fontes',
       3.0, 'informativo', 'TRATADO',
       'Interseção 2021-2023: ICMS 2015-2025, arrecadação 2016-2025, cadastro 2020-2025, desonerações 2021-2024, PIB 2018-2023. Cada pergunta declara sua própria janela',
       current_timestamp();

## 7. Cobertura dos relacionamentos

In [0]:
%sql

-- Chaves sem correspondência entre as tabelas

-- CNAEs do ICMS sem correspondência no de-para de cadeias
SELECT count(DISTINCT i.cnae_subclasse) AS cnaes_sem_cadeia,
       sum(i.valor_icms) AS valor_sem_cadeia
FROM silver.icms_cnae_subclasse i
LEFT JOIN (SELECT DISTINCT cnae_subclasse FROM silver.cnae_cadeia) c
  ON i.cnae_subclasse = c.cnae_subclasse
WHERE c.cnae_subclasse IS NULL AND NOT i.sem_cnae;

-- o mesmo para as desonerações detalhadas
SELECT count(DISTINCT d.cnae_subclasse) AS cnaes_sem_cadeia,
       sum(d.valor_desonerado) AS valor_sem_cadeia
FROM silver.desoneracoes d
LEFT JOIN (SELECT DISTINCT cnae_subclasse FROM silver.cnae_cadeia) c
  ON d.cnae_subclasse = c.cnae_subclasse
WHERE c.cnae_subclasse IS NULL AND d.nivel_agregacao = 'DETALHADO';

-- COREDEs das desonerações sem correspondência no cadastro
SELECT DISTINCT d.nome_corede
FROM silver.desoneracoes d
LEFT JOIN (SELECT DISTINCT nome_corede FROM silver.cadastro_municipio) m
  ON d.nome_corede = m.nome_corede
WHERE m.nome_corede IS NULL;

-- municípios da arrecadação sem código IBGE, fora os residuais
SELECT count(*) AS municipios_sem_ibge FROM silver.depara_municipio
WHERE cod_municipio_ibge IS NULL AND NOT pseudo_municipio;

In [0]:
%sql

-- Registro da cobertura das junções

INSERT INTO silver.relatorio_qualidade
SELECT 'Completude', 'icms_cnae_subclasse x cnae_cadeia',
       'percentual do ICMS com CNAE atribuído que não encontra cadeia',
       round(100 * coalesce(sum(CASE WHEN c.cnae_subclasse IS NULL THEN i.valor_icms END), 0)
             / sum(i.valor_icms), 3),
       'até 1%',
       CASE WHEN 100 * coalesce(sum(CASE WHEN c.cnae_subclasse IS NULL THEN i.valor_icms END), 0)
                 / sum(i.valor_icms) <= 1 THEN 'OK' ELSE 'ATENÇÃO' END,
       'De-para do Sebrae cobre a classificação quase por inteiro; 90 subclasses sem par, R$ 157,3 milhões na série',
       current_timestamp()
FROM silver.icms_cnae_subclasse i
LEFT JOIN (SELECT DISTINCT cnae_subclasse FROM silver.cnae_cadeia) c
  ON i.cnae_subclasse = c.cnae_subclasse
WHERE NOT i.sem_cnae;

INSERT INTO silver.relatorio_qualidade
SELECT 'Completude', 'desoneracoes x cnae_cadeia',
       'subclasses do nível detalhado sem cadeia correspondente',
       CAST(count(DISTINCT d.cnae_subclasse) AS DOUBLE), '0',
       CASE WHEN count(DISTINCT d.cnae_subclasse) = 0 THEN 'OK' ELSE 'ATENÇÃO' END,
       'Cobertura integral: toda subclasse com desoneração detalhada pertence a alguma cadeia',
       current_timestamp()
FROM silver.desoneracoes d
LEFT JOIN (SELECT DISTINCT cnae_subclasse FROM silver.cnae_cadeia) c
  ON d.cnae_subclasse = c.cnae_subclasse
WHERE c.cnae_subclasse IS NULL AND d.nivel_agregacao = 'DETALHADO';

INSERT INTO silver.relatorio_qualidade
SELECT 'Completude', 'arrecadacao_municipio x municipio_ibge',
       'municípios sem código IBGE, fora os dois residuais',
       CAST(count(*) AS DOUBLE), '0',
       CASE WHEN count(*) = 0 THEN 'OK' ELSE 'ATENÇÃO' END,
       'De-para resolvido por normalização de grafia; códigos 0 e 900 marcados como pseudo_municipio',
       current_timestamp()
FROM silver.depara_municipio
WHERE cod_municipio_ibge IS NULL AND NOT pseudo_municipio;

## 8. Cobertura de CNAEs por cadeia produtiva

Cadeias definidas por uma ou duas
subclasses não suportam conclusão sobre nível, e cadeias de serviço têm cobertura baixa porque o
ICMS incide sobre circulação de mercadorias, não sobre serviço tributado pelo ISS.

In [0]:
%sql

-- Cobertura de cada cadeia prioritária

WITH cnae_por_cadeia AS (
  SELECT c.cadeia, c.cnae_subclasse
  FROM silver.cnae_cadeia c WHERE c.cadeia_prioritaria
),
presenca AS (
  SELECT cc.cadeia,
         count(DISTINCT cc.cnae_subclasse) AS cnaes_da_cadeia,
         count(DISTINCT i.cnae_subclasse)  AS cnaes_com_icms
  FROM cnae_por_cadeia cc
  LEFT JOIN (SELECT DISTINCT cnae_subclasse FROM silver.icms_cnae_subclasse WHERE ano = 2024) i
    ON cc.cnae_subclasse = i.cnae_subclasse
  GROUP BY cc.cadeia
)
SELECT cadeia, cnaes_da_cadeia, cnaes_com_icms,
       round(100 * cnaes_com_icms / cnaes_da_cadeia, 1) AS cobertura_perc
FROM presenca ORDER BY cnaes_da_cadeia;

In [0]:
%sql

-- Registro das cadeias frágeis e das de baixa cobertura

INSERT INTO silver.relatorio_qualidade
SELECT 'Acurácia', 'cnae_cadeia', 'cadeias com menos de 3 CNAEs',
       CAST(count(*) AS DOUBLE), 'informativo', 'TRATADO',
       'Cadeias definidas por uma ou duas subclasses não suportam conclusão sobre nível; nas análises são sinalizadas',
       current_timestamp()
FROM (SELECT cadeia FROM silver.cnae_cadeia WHERE cadeia_prioritaria
      GROUP BY cadeia HAVING count(DISTINCT cnae_subclasse) < 3);

INSERT INTO silver.relatorio_qualidade
SELECT 'Acurácia', 'cnae_cadeia x icms_cnae_subclasse',
       'cadeias prioritárias com cobertura de ICMS abaixo de 70%',
       CAST(count(*) AS DOUBLE), 'informativo', 'TRATADO',
       'Saúde 33,8%, Horticultura 51,3%, Turismo 62,1% e Pecuária 68,8%: cadeias de serviço e produção primária, tributadas por ISS ou sob isenção. Não é falta de dado; o ICMS mede fração dessas cadeias e os resultados levam ressalva',
       current_timestamp()
FROM (
  SELECT c.cadeia
  FROM silver.cnae_cadeia c
  LEFT JOIN (SELECT DISTINCT cnae_subclasse FROM silver.icms_cnae_subclasse WHERE ano = 2024) i
    ON c.cnae_subclasse = i.cnae_subclasse
  WHERE c.cadeia_prioritaria
  GROUP BY c.cadeia
  HAVING 100 * count(DISTINCT i.cnae_subclasse) / count(DISTINCT c.cnae_subclasse) < 70
);

##9. Reconciliação entre camadas

Comparação entre Bronze e Silver, tabela a tabela.

In [0]:
%sql

-- Reconciliação entre camadas: a transformação não pode criar nem perder dado

CREATE OR REPLACE TEMP VIEW reconciliacao AS
SELECT 'icms_cnae_subclasse' AS tabela,
       (SELECT count(*) FROM bronze.icms_cnae_subclasse) AS linhas_bronze,
       (SELECT count(*) FROM silver.icms_cnae_subclasse) AS linhas_silver,
       (SELECT sum(CAST(replace(replace(valor, '.', ''), ',', '.') AS DECIMAL(18,2)))
        FROM bronze.icms_cnae_subclasse) AS medida_bronze,
       (SELECT sum(valor_icms) FROM silver.icms_cnae_subclasse) AS medida_silver
UNION ALL
SELECT 'arrecadacao_municipio',
       (SELECT count(*) FROM bronze.arrecadacao_municipio_corede),
       (SELECT count(*) FROM silver.arrecadacao_municipio),
       (SELECT sum(CAST(replace(replace(valor, '.', ''), ',', '.') AS DECIMAL(18,2)))
        FROM bronze.arrecadacao_municipio_corede),
       (SELECT sum(valor_arrecadado) FROM silver.arrecadacao_municipio)
UNION ALL
SELECT 'desoneracoes',
       (SELECT count(*) FROM (SELECT 1 FROM bronze.desoneracoes_2021_2023
                              UNION ALL SELECT 1 FROM bronze.desoneracoes_2024)),
       (SELECT count(*) FROM silver.desoneracoes),
       (SELECT sum(CAST(replace(replace(vlr_desoner, '.', ''), ',', '.') AS DECIMAL(18,2)))
        FROM (SELECT vlr_desoner FROM bronze.desoneracoes_2021_2023
              UNION ALL SELECT vlr_desoner FROM bronze.desoneracoes_2024)),
       (SELECT sum(valor_desonerado) FROM silver.desoneracoes)
UNION ALL
SELECT 'cadastro_setor',
       (SELECT count(*) FROM bronze.cadastro_contribuintes_setor),
       (SELECT count(*) FROM silver.cadastro_setor),
       (SELECT sum(CAST(estabelecimentos_ativos AS DECIMAL(18,2)))
        FROM bronze.cadastro_contribuintes_setor),
       (SELECT sum(CAST(qtd_ativos AS DECIMAL(18,2))) FROM silver.cadastro_setor)
UNION ALL
SELECT 'cadastro_municipio',
       (SELECT count(*) FROM bronze.cadastro_contribuintes_municipio),
       (SELECT count(*) FROM silver.cadastro_municipio),
       (SELECT sum(CAST(qtd_ativos AS DECIMAL(18,2)))
        FROM bronze.cadastro_contribuintes_municipio),
       (SELECT sum(CAST(qtd_ativos AS DECIMAL(18,2))) FROM silver.cadastro_municipio)
UNION ALL
SELECT 'municipio_ibge',
       (SELECT count(*) FROM bronze.ibge_municipios_rs),
       (SELECT count(*) FROM silver.municipio_ibge),
       CAST(NULL AS DECIMAL(18,2)), CAST(NULL AS DECIMAL(18,2));

SELECT tabela, linhas_bronze, linhas_silver,
       linhas_silver - linhas_bronze AS dif_linhas,
       medida_bronze, medida_silver,
       round(medida_silver - medida_bronze, 2) AS dif_medida
FROM reconciliacao ORDER BY tabela;

In [0]:
%sql

INSERT INTO silver.relatorio_qualidade
SELECT 'Reconciliação', tabela, 'diferença de linhas entre Bronze e Silver',
       CAST(linhas_silver - linhas_bronze AS DOUBLE), '0',
       CASE WHEN linhas_silver = linhas_bronze THEN 'OK' ELSE 'ATENÇÃO' END,
       'Transformação não pode criar nem perder linha', current_timestamp()
FROM reconciliacao
UNION ALL
SELECT 'Reconciliação', tabela, 'diferença percentual da medida entre Bronze e Silver',
       round(100 * (medida_silver - medida_bronze) / medida_bronze, 6), '0',
       CASE WHEN abs(medida_silver - medida_bronze) < 0.01 THEN 'OK' ELSE 'ATENÇÃO' END,
       'Conversão de decimal com vírgula e tipagem preservam o valor', current_timestamp()
FROM reconciliacao WHERE medida_bronze IS NOT NULL;

## 9. Relatório consolidado

`OK` indica critério atendido; `TRATADO` indica problema conhecido, medido e com decisão registrada na observação.

In [0]:
%sql

SELECT dimensao, tabela, verificacao, valor, esperado, situacao, observacao
FROM silver.relatorio_qualidade
ORDER BY CASE dimensao
           WHEN 'Completude'   THEN 1
           WHEN 'Consistência' THEN 2
           WHEN 'Unicidade'    THEN 3
           WHEN 'Acurácia'      THEN 4
           WHEN 'Outliers'      THEN 5
           WHEN 'Reconciliação' THEN 6 END,
         tabela;

In [0]:
%sql

-- Resumo por situação

SELECT situacao, count(*) AS verificacoes
FROM silver.relatorio_qualidade GROUP BY situacao;